##Phase 1: The Deterministic Engine (Data Engineering)

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('shipment_records.csv')

df['shipment_date'] = pd.to_datetime(df['shipment_date'])
df['week_of'] = df['shipment_date'] - pd.to_timedelta(df['shipment_date'].dt.weekday, unit='D')
df['route'] = df['origin'] + '-' + df['destination']

weekly_route_data = df.groupby(['route', 'route_type', 'week_of']).agg(
    total_cost=('freight_cost_inr', 'sum'),
    total_tonne_km=('quantity_tonnes', lambda q: (q * df.loc[q.index, 'distance_km']).sum())
).reset_index()

weekly_route_data['cost_per_tonne_km'] = weekly_route_data['total_cost'] / weekly_route_data['total_tonne_km']

weekly_route_data = weekly_route_data.sort_values(by=['route', 'week_of']).reset_index(drop=True)

weekly_route_data['historical_avg_8w'] = weekly_route_data.groupby('route')['cost_per_tonne_km'].transform(
    lambda x: x.shift(1).rolling(window=8, min_periods=1).mean()
)

def calculate_peer_avg(row, df_all):
    peers = df_all[(df_all['week_of'] == row['week_of']) &
                   (df_all['route_type'] == row['route_type']) &
                   (df_all['route'] != row['route'])]
    return peers['cost_per_tonne_km'].mean()

weekly_route_data['peer_avg_this_week'] = weekly_route_data.apply(lambda row: calculate_peer_avg(row, weekly_route_data), axis=1)

THRESHOLD = 1.10

weekly_route_data['is_spike'] = (
    (weekly_route_data['cost_per_tonne_km'] > (weekly_route_data['historical_avg_8w'] * THRESHOLD)) |
    (weekly_route_data['cost_per_tonne_km'] > (weekly_route_data['peer_avg_this_week'] * THRESHOLD))
)

print(f"Total weeks processed: {len(weekly_route_data)}")
print(f"Spikes flagged for AI review: {weekly_route_data['is_spike'].sum()}")
display(weekly_route_data[weekly_route_data['is_spike'] == True].head())

Total weeks processed: 728
Spikes flagged for AI review: 90


,route,route_type,week_of,total_cost,total_tonne_km,cost_per_tonne_km,historical_avg_8w,peer_avg_this_week,is_spike
55,Ahmedabad-Mumbai,Medium,2025-01-20,134631,40916.71,3.290367,2.541352,2.686152,True
128,Chennai-Bangalore,Medium,2024-06-17,50991,18176.68,2.805298,2.675859,2.546169,True
134,Chennai-Bangalore,Medium,2024-07-29,65629,23658.24,2.774044,2.712592,2.511577,True
136,Chennai-Bangalore,Medium,2024-08-12,44144,15802.03,2.793565,2.715689,2.500641,True
163,Chennai-Bangalore,Medium,2025-02-17,59877,21061.88,2.842909,2.714097,2.576805,True


##Phase 2: The RAG System

In [2]:
!pip install sentence-transformers google-generativeai scikit-learn

In [4]:
!pip uninstall -y google-generativeai
!pip install google-genai sentence-transformers scikit-learn

Found existing installation: google-generativeai 0.8.6
Uninstalling google-generativeai-0.8.6:
  Successfully uninstalled google-generativeai-0.8.6


In [29]:
"""
================================================================================
FREIGHT COST ANOMALY AUDITOR - PHASE 2 (RAG & LLM JUDGE)
================================================================================
1. Deterministic Phase 1 groups records and flags spikes using historical/peer baselines.
2. Hybrid Retrieval filters context notes strictly by route and date window, then ranks by embeddings.
3. Constrained LLM Judge reads surviving candidates and determines if a cost rise is justified.
4. Code-level Validation prevents hallucinations and chronologically impossible justifications.
5. Summary Report and automated assertions ensure reproducible, trusted outcomes.
================================================================================
"""

import os
import json
import time
import pandas as pd
import numpy as np
from datetime import timedelta
from google import genai
from google.genai import types
from google.colab import userdata
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

# ==============================================================================
# A. CONFIGURATION BLOCK
# ==============================================================================
TEST_N = None          # Set to an int (e.g., 5) for smoke testing, or None for all rows
RESET = True           # Set to True to wipe the flawed checkpoint and run cleanly
CKPT_FILE = "spike_results.json"
WINDOW_DAYS = 21       # Lookback window for notes (days before the spike week)
TOP_K = 3              # Max notes to send to the LLM

MODEL_CHAIN = [
    'gemini-3.5-flash',
    'gemini-3.1-flash-lite',
    'gemini-3.5-flash-lite',
    'gemini-3-flash-preview'
]

# ==============================================================================
# B. SETUP & DATA PREPARATION
# ==============================================================================
# 1. Init Client securely
try:
    api_key = userdata.get('GEMINI_API_KEY')
    client = genai.Client(api_key=api_key)
except Exception as e:
    raise RuntimeError(f"Failed to initialize GenAI client. Ensure GEMINI_API_KEY is in Colab Secrets. {e}")

# 2. Embeddings & Notes setup
print("Loading embedding model and context notes...")
embedder = SentenceTransformer('all-MiniLM-L6-v2')

notes_df = pd.read_csv('context_notes.csv')
notes_df['date'] = pd.to_datetime(notes_df['date'])
# Pre-compute search text and embeddings
notes_df['search_text'] = "Route: " + notes_df['applies_to'] + " | Date: " + notes_df['date'].dt.strftime('%Y-%m-%d') + " | Note: " + notes_df['note']
note_embeddings = embedder.encode(notes_df['search_text'].tolist())

# 3. Resume State (Checkpointing)
if RESET and os.path.exists(CKPT_FILE):
    os.remove(CKPT_FILE)
    print("Checkpoint reset.")

if os.path.exists(CKPT_FILE):
    with open(CKPT_FILE, 'r') as f:
        results_ckpt = json.load(f)
    print(f"Resuming from checkpoint with {len(results_ckpt)} processed rows.")
else:
    results_ckpt = {}

# Dictionary to track models that have exhausted retries (5 min cooldown)
model_cooldowns = {}

# Define strict JSON schema for LLM
response_schema = types.Schema(
    type=types.Type.OBJECT,
    properties={
        "flagged": types.Schema(type=types.Type.STRING, enum=["Yes", "No (justified)"]),
        "matched_note_id": types.Schema(type=types.Type.STRING),
        "reason": types.Schema(type=types.Type.STRING),
    },
    required=["flagged", "matched_note_id", "reason"]
)

# ==============================================================================
# C. HELPER FUNCTIONS
# ==============================================================================
def get_hybrid_candidates(route, week_of_ts):
    """Deterministic filter (Route + Date) -> Vector Rank -> Top K"""
    week_end_ts = week_of_ts + pd.Timedelta(days=6)
    lookback_ts = week_of_ts - pd.Timedelta(days=WINDOW_DAYS)

    # 1. Deterministic Filter
    mask_route = notes_df['applies_to'].isin([route, 'All Routes'])
    mask_date = (notes_df['date'] >= lookback_ts) & (notes_df['date'] <= week_end_ts)
    valid_idx = notes_df.index[mask_route & mask_date].tolist()

    if not valid_idx:
        return pd.DataFrame()

    # 2. Vector Rank
    query = f"Route: {route} | Date: {week_of_ts.strftime('%Y-%m-%d')} | Looking for cost increase reasons."
    q_emb = embedder.encode([query])

    valid_embeddings = note_embeddings[valid_idx]
    sims = cosine_similarity(q_emb, valid_embeddings)[0]

    # Sort descending
    ranked_local_indices = sims.argsort()[::-1][:TOP_K]
    top_global_indices = [valid_idx[i] for i in ranked_local_indices]

    return notes_df.loc[top_global_indices]

def call_llm_with_resilience(prompt):
    """Attempts LLM calls falling down the MODEL_CHAIN, with exponential backoff and cooldowns."""
    max_retries = 3

    for model_name in MODEL_CHAIN:
        # Check cooldown
        if model_name in model_cooldowns:
            if time.time() < model_cooldowns[model_name]:
                continue
            else:
                del model_cooldowns[model_name] # Cooldown expired

        for attempt in range(max_retries):
            try:
                resp = client.models.generate_content(
                    model=model_name,
                    contents=prompt,
                    config=types.GenerateContentConfig(
                        temperature=0.0,
                        response_mime_type="application/json",
                        response_schema=response_schema
                    )
                )
                text_val = resp.text
                if text_val.startswith('```json'): text_val = text_val[7:-3]

                return json.loads(text_val), model_name

            except Exception as e:
                err_str = str(e)
                print(f"    [!] {model_name} attempt {attempt+1} failed: {type(e).__name__} - {err_str[:80]}")

                if "404" in err_str:
                    break # Dead/retired model. Immediately move to next model in chain.

                # Exponential backoff for 429, 503, 500
                time.sleep(2 ** attempt)
        else:
            print(f"    [!] {model_name} exhausted retries. Initiating 5-minute cooldown.")
            model_cooldowns[model_name] = time.time() + 300

    return None, None # All models failed

# ==============================================================================
# D. MAIN EXECUTION LOOP
# ==============================================================================
# Calculate derived columns (overruns)
weekly_route_data['overrun_vs_history_pct'] = ((weekly_route_data['cost_per_tonne_km'] - weekly_route_data['historical_avg_8w']) / weekly_route_data['historical_avg_8w'] * 100).round(2)
weekly_route_data['overrun_vs_peers_pct'] = ((weekly_route_data['cost_per_tonne_km'] - weekly_route_data['peer_avg_this_week']) / weekly_route_data['peer_avg_this_week'] * 100).round(2)

spikes_df = weekly_route_data[weekly_route_data['is_spike'] == True].copy()
if TEST_N is not None:
    spikes_df = spikes_df.head(TEST_N)

print(f"\nProcessing {len(spikes_df)} spikes...")

for index, row in spikes_df.iterrows():
    # Unique ID for checkpointing
    row_key = f"{row['route']}_{row['week_of'].strftime('%Y-%m-%d')}"

    # RESUME SAFE: skip if already processed successfully or deterministically bypassed
    if row_key in results_ckpt and results_ckpt[row_key].get('status') in ['ok', 'rule']:
        continue

    print(f"Evaluating {row['route']} for week {row['week_of'].strftime('%Y-%m-%d')}...")

    # 1. Hybrid Retrieval
    candidates = get_hybrid_candidates(row['route'], row['week_of'])

    if candidates.empty:
        results_ckpt[row_key] = {
            "flagged": "Yes",
            "matched_note_id": "",
            "reason": "No note matches this route and timeframe.",
            "model": "rule",
            "status": "rule"
        }
    else:
        # 2. Build Context
        context_str = ""
        candidate_ids = set()
        for _, n in candidates.iterrows():
            context_str += f"- ID: {n['note_id']} | Date: {n['date'].strftime('%Y-%m-%d')} | Route: {n['applies_to']} | Text: {n['note']}\n"
            candidate_ids.add(n['note_id'])

        prompt = f"""
        You are an AI auditor for a freight company. A route's cost has spiked.
        Determine if the spike is justified based ONLY on the context notes provided.

        Spike Details:
        - Route: {row['route']}
        - Week Of (Monday): {row['week_of'].strftime('%Y-%m-%d')}
        - Current Cost per tonne-km: {row['cost_per_tonne_km']:.2f}
        - Historical Avg: {row['historical_avg_8w']:.2f}
        - Peer Avg: {row['peer_avg_this_week']:.2f}

        Retrieved Context Notes:
        {context_str}

        Rules:
        1. The note MUST apply to the route (or 'All Routes').
        2. The period described IN THE NOTE TEXT must cover the spike week.
        3. The note MUST explain a cost INCREASE (not "returned to normal", "not affected", "no rate change", "stable").
        4. If no note perfectly matches, it is NOT justified (flagged: Yes). DO NOT HALLUCINATE.
        """

        # 3. Call LLM
        llm_result, used_model = call_llm_with_resilience(prompt)

        if llm_result is None:
            # API failure - record as error, do NOT classify as unexplained spike.
            results_ckpt[row_key] = {
                "flagged": "ERROR", "matched_note_id": "", "reason": "API Failure across model chain.",
                "model": "none", "status": "error"
            }
        else:
            # 4. Anti-Hallucination & Label Drift Post-Validation
            flagged = llm_result.get('flagged', 'Yes')
            if flagged not in ["Yes", "No (justified)"]:
                flagged = "Yes" # Normalise label drift

            note_id = llm_result.get('matched_note_id', '')
            reason = llm_result.get('reason', '')
            original_reason = None

            if flagged == "No (justified)":
                if note_id not in candidate_ids:
                    flagged, original_reason = "Yes", reason
                    reason = f"OVERRIDE: LLM cited a note ({note_id}) not in retrieved candidates."
                    note_id = ""
                else:
                    note_dt = candidates.loc[candidates['note_id'] == note_id, 'date'].iloc[0]
                    week_end = row['week_of'] + pd.Timedelta(days=6)
                    if note_dt > week_end:
                        flagged, original_reason = "Yes", reason
                        reason = f"OVERRIDE: Cited note {note_id} is dated after the spike week end."
                        note_id = ""

            # ENFORCED GUARDRAIL: If it's flagged as unexplained, wipe the note ID.
            if flagged == "Yes":
                note_id = ""

            results_ckpt[row_key] = {
                "flagged": flagged,
                "matched_note_id": note_id,
                "reason": reason,
                "model": used_model,
                "status": "ok"
            }
            if original_reason:
                results_ckpt[row_key]['original_reason'] = original_reason

            # Respect rate limits on successful calls
            time.sleep(1)

    # Save checkpoint after every row
    with open(CKPT_FILE, 'w') as f:
        json.dump(results_ckpt, f)

# ==============================================================================
# E. FINALISE AND EXPORT
# ==============================================================================
# Map results back to spikes
def map_ckpt(row, field, default_val=np.nan):
    k = f"{row['route']}_{row['week_of'].strftime('%Y-%m-%d')}"
    return results_ckpt.get(k, {}).get(field, default_val)

spikes_df['flagged'] = spikes_df.apply(lambda r: map_ckpt(r, 'flagged', 'UNPROCESSED'), axis=1)
spikes_df['matched_note_id'] = spikes_df.apply(lambda r: map_ckpt(r, 'matched_note_id', ''), axis=1)
spikes_df['reason'] = spikes_df.apply(lambda r: map_ckpt(r, 'reason', ''), axis=1)
spikes_df['model'] = spikes_df.apply(lambda r: map_ckpt(r, 'model', ''), axis=1)
spikes_df['status'] = spikes_df.apply(lambda r: map_ckpt(r, 'status', 'unprocessed'), axis=1)

# Sort by severity (highest overrun vs history)
final = spikes_df.sort_values(by='overrun_vs_history_pct', ascending=False).reset_index(drop=True)

# Select and order final columns
cols = ['route', 'route_type', 'week_of', 'cost_per_tonne_km', 'historical_avg_8w',
        'peer_avg_this_week', 'overrun_vs_history_pct', 'overrun_vs_peers_pct',
        'flagged', 'matched_note_id', 'reason', 'model', 'status']
final = final[cols]

final.to_csv('spike_audit_results.csv', index=False)

# ==============================================================================
# F. SUMMARY REPORT & AUTOMATED ASSERTIONS
# ==============================================================================
print("\n" + "="*50)
print("PHASE 2 SUMMARY REPORT")
print("="*50)
print(f"Total Spikes Analyzed : {len(final)}")
print(f"Statuses              :\n{final['status'].value_counts().to_string()}")
print(f"Flagged Outcomes      :\n{final['flagged'].value_counts().to_string()}")
print(f"Processing Engine     :\n{final['model'].value_counts().to_string()}")
print(f"Zero-LLM calls saved  : {len(final[final['model'] == 'rule'])}")
cited = final[final['matched_note_id'] != '']['matched_note_id'].value_counts()
print(f"Notes Cited           :\n{cited.to_string() if not cited.empty else 'None'}")
print("="*50)

print("\nRunning Edge-Case Assertions:")
def check_case(route, date_str, exp_flagged, exp_note=None, forbidden_notes=None):
    row = final[(final['route'] == route) & (final['week_of'] == pd.to_datetime(date_str))]
    if row.empty: return # Only check if it exists in the data

    flagged = row['flagged'].iloc[0]
    note = row['matched_note_id'].iloc[0]

    passed = flagged == exp_flagged
    if exp_note: passed = passed and (note == exp_note)
    if forbidden_notes: passed = passed and (note not in forbidden_notes)

    print(f"[{'PASS' if passed else 'FAIL'}] {route} ({date_str}): Expected {exp_flagged}" +
          (f" citing {exp_note}" if exp_note else ""))

# Run specific edge cases
check_case('Chennai-Bangalore', '2025-02-17', 'Yes')
check_case('Chennai-Bangalore', '2025-02-24', 'No (justified)', 'N001')
check_case('Chennai-Bangalore', '2025-03-03', 'No (justified)', 'N001')
check_case('Chennai-Bangalore', '2025-03-10', 'Yes')
check_case('Chennai-Bangalore', '2025-03-17', 'Yes')
check_case('Ahmedabad-Mumbai',  '2025-05-05', 'No (justified)', 'N003')
check_case('Mumbai-Pune',       '2025-04-28', 'Yes')

# Distractor check across all rows
distractors = ['N004', 'N006', 'N010']
bad_cites = final[final['matched_note_id'].isin(distractors)]
print(f"[{'PASS' if bad_cites.empty else 'FAIL'}] Distractor notes (N004, N006, N010) strictly avoided.")

print("\nPhase 2 Complete. File saved to 'spike_audit_results.csv'.")
print("To retry errors ONLY, ensure RESET=False and run this cell again.")

Loading embedding model and context notes...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Checkpoint reset.

Processing 90 spikes...
Evaluating Ahmedabad-Mumbai for week 2025-01-20...
    [!] gemini-3.5-flash attempt 1 failed: ClientError - 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your cu
    [!] gemini-3.5-flash attempt 2 failed: ServerError - 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently exp
    [!] gemini-3.5-flash attempt 3 failed: ServerError - 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently exp
    [!] gemini-3.5-flash exhausted retries. Initiating 5-minute cooldown.
Evaluating Chennai-Bangalore for week 2024-06-17...
Evaluating Chennai-Bangalore for week 2024-07-29...
Evaluating Chennai-Bangalore for week 2024-08-12...
Evaluating Chennai-Bangalore for week 2025-02-17...
Evaluating Chennai-Bangalore for week 2025-02-24...
Evaluating Chennai-Bangalore for week 2025-03-03...
Evaluating Chennai-Bangalore for week 2025-03-10...
Evaluating Chennai-Bangalore for week 2025-03-17...
Ev

## Phase 3

In [30]:
import pandas as pd

# 1. Load the successful results from Phase 2
results = pd.read_csv('spike_audit_results.csv')

# 2. Format the percentage strings exactly as the autograder expects
def format_pct(val, suffix):
    if pd.isna(val):
        return ""
    sign = "+" if val >= 0 else ""
    return f"{sign}{val:.1f}% {suffix}"

results['vs_own_history'] = results['overrun_vs_history_pct'].apply(
    lambda x: format_pct(x, "vs this route's past average")
)
results['vs_similar_routes'] = results['overrun_vs_peers_pct'].apply(
    lambda x: format_pct(x, "vs similar-length routes this week")
)

# 3. Select and order columns to match the strict contract
final_cols = [
    'route',
    'week_of',
    'cost_per_tonne_km',
    'vs_own_history',
    'vs_similar_routes',
    'flagged',
    'matched_note_id',
    'reason'
]
final_output = results[final_cols]

# 4. Save the final CSV for submission
final_output.to_csv('final_submission.csv', index=False)
print("✅ Contract matched! 'final_submission.csv' saved and ready for upload.\n")

# 5. Display the table so you can verify it
display(final_output.head())

✅ Contract matched! 'final_submission.csv' saved and ready for upload.



,route,week_of,cost_per_tonne_km,vs_own_history,vs_similar_routes,flagged,matched_note_id,reason
0,Delhi-Jaipur,2024-11-11,4.172957,+35.5% vs this route's past average,+21.0% vs similar-length routes this week,Yes,NaN,No note matches this route and timeframe.
1,Chennai-Bangalore,2025-02-24,3.582827,+31.5% vs this route's past average,+38.7% vs similar-length routes this week,No (justified),N001,"The cost spike is justified by note N001, which reports heavy flooding on the Chennai-Bangalore route during the week of 2025-02-24, necessitating longer detours and resulting in higher trip costs."
2,Ahmedabad-Mumbai,2025-01-20,3.290367,+29.5% vs this route's past average,+22.5% vs similar-length routes this week,No (justified),N002,"The cost spike is justified by note N002, which explicitly mentions a temporary surcharge on the Ahmedabad-Mumbai corridor for the week of 2025-01-20 due to high demand and limited truck availability during a regional festival."
3,Delhi-Jaipur,2024-11-18,4.090564,+27.6% vs this route's past average,+19.7% vs similar-length routes this week,Yes,NaN,No note matches this route and timeframe.
4,Chennai-Bangalore,2025-03-03,3.476189,+22.8% vs this route's past average,+32.0% vs similar-length routes this week,No (justified),N001,"The note explicitly mentions heavy flooding on the Chennai-Bangalore route causing longer detours and higher trip costs during the period of Feb 24 to Mar 8, which directly covers the spike week of 2025-03-03."
